In [10]:
import pandas as pd
import numpy as np
import hnswlib as hl
import hnsw_benchmark as bm
import twonn
import farhest_point_order as fp
import matplotlib.pyplot as plt
import os

In [11]:
# Alapadatok megadása
dataset_names = [
    "arguana",
    "nfcorpus"
]
embedding_models = [
    "e5-small-v2",
    "multilingual-e5-large",
    "bge-small-en",
    "stella-base-en-v2",
    "all-MiniLM-L6-v2"
]
options = [
    # "_lid_fps_lpca_2nn",
    # "_lid_fps",
    "_lid",
    "_lid",
    "_lid",
    "_lid",
    "_lid"
]

df_corpus = {}
for dataset_name in dataset_names:
    for embedding_model, option in zip(embedding_models, options):
        corpus_filename = dataset_name + "_corpus_" + embedding_model + option

        if dataset_name == 'arguana': # kell, ki kell majd venni
            if embedding_model == 'e5-small-v2':
                corpus_filename = corpus_filename + "_fps_lpca_2nn"
            if embedding_model == 'multilingual-e5-large':
                corpus_filename = corpus_filename + "_fps"

        corpus_input_path = os.path.join("..\\data\\", corpus_filename + ".json")
        corpus_data = pd.read_json(corpus_input_path, orient="records", lines=True)
        df_corpus[dataset_name + "_" + embedding_model] = corpus_data

df_query = {}
for dataset_name in dataset_names:
    for embedding_model in embedding_models:
        query_filename = dataset_name + "_query_" + embedding_model
        query_input_path = os.path.join("..\\data\\", query_filename + ".json")
        query_data = pd.read_json(query_input_path, orient="records", lines=True)
        df_query[dataset_name + "_" + embedding_model] = query_data

In [12]:
for name in df_corpus:
    print("-- " + name + " --")

    print("Corpus data")
    corpus_vectors = np.asarray([np.asarray(v) for v in df_corpus[name].vectors], dtype=np.float32)
    print(corpus_vectors.shape)
    print(df_corpus[name].keys())

    print("Query data")
    query_vectors = np.asarray([np.asarray(v) for v in df_query[name].vectors], dtype=np.float32)
    print(query_vectors.shape)

    print()


-- arguana_e5-small-v2 --
Corpus data
(8674, 384)
Index(['vectors', 'LID', 'fp_score', 'LPCA_dim', 'TwoNN'], dtype='str')
Query data
(1406, 384)

-- arguana_multilingual-e5-large --
Corpus data
(8674, 1024)
Index(['vectors', 'LID', 'fp_score'], dtype='str')
Query data
(1406, 1024)

-- arguana_bge-small-en --
Corpus data
(8674, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 384)

-- arguana_stella-base-en-v2 --
Corpus data
(8674, 768)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 768)

-- arguana_all-MiniLM-L6-v2 --
Corpus data
(8674, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(1406, 384)

-- nfcorpus_e5-small-v2 --
Corpus data
(3633, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(3237, 384)

-- nfcorpus_multilingual-e5-large --
Corpus data
(3633, 1024)
Index(['vectors', 'LID'], dtype='str')
Query data
(3237, 1024)

-- nfcorpus_bge-small-en --
Corpus data
(3633, 384)
Index(['vectors', 'LID'], dtype='str')
Query data
(3237, 384)

-- nfcorpus

In [61]:
# Benchmarkok futtatása
def run_benchmark(corpus_data, sorted_indices, query_data, n_times=1, ef_search=10):
    recalls = []
    for _ in range(0, n_times):
        bm_data = [corpus_data[i] for i in sorted_indices]
        benchmark = bm.HNSW_benchmark(data=bm_data)
        recall_list = benchmark.run(qs=query_data, ef_search=ef_search)
        recalls.append(np.mean(recall_list))

    print(f"Minimum recall: {np.min(recalls):.4f}")
    print(f"Average recall: {np.mean(recalls):.4f}")
    print(f"Maximum recall: {np.max(recalls):.4f}")

    # plt.figure(figsize=(8, 5))
    # plt.hist(recalls, bins=10, edgecolor="black", alpha=0.75)
    # plt.title("Distribution of recall values")
    # plt.xlabel("Recall")
    # plt.ylabel("Count")
    # plt.grid(axis="y", alpha=0.3)
    # plt.ylim(0, len(df_query.vectors))
    # plt.show()
    return recalls

In [70]:
# HNSW benchmark - alap beállítások
n_times = 100
ef_search = 10

df_bm = pd.DataFrame(columns=[
    "em",
    "dn",
    "random",
    "lid_d",
    "lid_a"
])

# df_bm2 = pd.DataFrame(columns=[
#     "em",
#     "random",
#     "lid_d"
# ])

In [71]:
# HNSW benchmark - Random beszúrás
print("-- Random --")
print()
print()
for em in embedding_models:
    print("-- " + em + " --")
    rcs = []
    for dn in dataset_names:
        print("  -- " + dn + " --")

        sorted_indices = np.arange(len(df_corpus[dn + "_" + em].vectors))
        rc = run_benchmark(df_corpus[dn + "_" + em].vectors, sorted_indices, df_query[dn + "_" + em].vectors, n_times, ef_search)
        df_bm.loc[len(df_bm)] = [em, dn, np.mean(rc), .0, .0]
        rcs.append(rc)

        print()
    print(f"Avg recall: {np.mean(rcs):.4f}")
    print()
    

-- Random --


-- e5-small-v2 --
  -- arguana --
Minimum recall: 0.9067
Average recall: 0.9142
Maximum recall: 0.9292

  -- nfcorpus --
Minimum recall: 0.6920
Average recall: 0.6976
Maximum recall: 0.7064

Avg recall: 0.8059

-- multilingual-e5-large --
  -- arguana --
Minimum recall: 0.9033
Average recall: 0.9152
Maximum recall: 0.9229

  -- nfcorpus --
Minimum recall: 0.6970
Average recall: 0.7042
Maximum recall: 0.7114

Avg recall: 0.8097

-- bge-small-en --
  -- arguana --
Minimum recall: 0.9132
Average recall: 0.9209
Maximum recall: 0.9291

  -- nfcorpus --
Minimum recall: 0.6313
Average recall: 0.6438
Maximum recall: 0.6660

Avg recall: 0.7824

-- stella-base-en-v2 --
  -- arguana --
Minimum recall: 0.9086
Average recall: 0.9164
Maximum recall: 0.9279

  -- nfcorpus --
Minimum recall: 0.7371
Average recall: 0.7426
Maximum recall: 0.7503

Avg recall: 0.8295

-- all-MiniLM-L6-v2 --
  -- arguana --
Minimum recall: 0.9496
Average recall: 0.9560
Maximum recall: 0.9617

  -- nfcorpus -

In [72]:
# HNSW benchmark - LID szerint csökkenő
print("-- Descending by LID --")
print()
print()
for em in embedding_models:
    print("-- " + em + " --")
    rcs = []
    for dn in dataset_names:
        print("  -- " + dn + " --")

        sorted_indices = np.argsort(df_corpus[dn + "_" + em].LID, descending=True)
        rc = run_benchmark(df_corpus[dn + "_" + em].vectors, sorted_indices, df_query[dn + "_" + em].vectors, n_times, ef_search)
        # df_bm.loc[len(df_bm)] = [em, dn, np.mean(rc), 0]
        # df_bm.loc[(df_bm.em == em and df_bm.dn == dn), 'lid_d'] = np.mean(rc)
        upd = df_bm.em == em
        upd &= df_bm.dn == dn
        df_bm.loc[upd, 'lid_d'] = np.mean(rc)
        rcs.append(rc)

        print()
    print(f"Avg recall: {np.mean(rcs):.4f}")
    print()

-- Descending by LID --


-- e5-small-v2 --
  -- arguana --
Minimum recall: 0.9464
Average recall: 0.9497
Maximum recall: 0.9537

  -- nfcorpus --
Minimum recall: 0.7414
Average recall: 0.7443
Maximum recall: 0.7483

Avg recall: 0.8470

-- multilingual-e5-large --
  -- arguana --
Minimum recall: 0.9539
Average recall: 0.9570
Maximum recall: 0.9600

  -- nfcorpus --
Minimum recall: 0.7458
Average recall: 0.7497
Maximum recall: 0.7528

Avg recall: 0.8533

-- bge-small-en --
  -- arguana --
Minimum recall: 0.9620
Average recall: 0.9656
Maximum recall: 0.9689

  -- nfcorpus --
Minimum recall: 0.7532
Average recall: 0.7570
Maximum recall: 0.7639

Avg recall: 0.8613

-- stella-base-en-v2 --
  -- arguana --
Minimum recall: 0.9565
Average recall: 0.9607
Maximum recall: 0.9643

  -- nfcorpus --
Minimum recall: 0.8031
Average recall: 0.8077
Maximum recall: 0.8116

Avg recall: 0.8842

-- all-MiniLM-L6-v2 --
  -- arguana --
Minimum recall: 0.9856
Average recall: 0.9876
Maximum recall: 0.9895

  --

In [73]:
# HNSW benchmark - LID szerint növekvő
print("-- Ascending by LID --")
print()
print()
for em in embedding_models:
    print("-- " + em + " --")
    rcs = []
    for dn in dataset_names:
        print("  -- " + dn + " --")

        sorted_indices = np.argsort(df_corpus[dn + "_" + em].LID, descending=False)
        rc = run_benchmark(df_corpus[dn + "_" + em].vectors, sorted_indices, df_query[dn + "_" + em].vectors, n_times, ef_search)
        upd = df_bm.em == em
        upd &= df_bm.dn == dn
        df_bm.loc[upd, 'lid_a'] = np.mean(rc)
        rcs.append(rc)

        print()
    print(f"Avg recall: {np.mean(rcs):.4f}")
    print()

-- Ascending by LID --


-- e5-small-v2 --
  -- arguana --
Minimum recall: 0.9286
Average recall: 0.9334
Maximum recall: 0.9381

  -- nfcorpus --
Minimum recall: 0.6895
Average recall: 0.6921
Maximum recall: 0.6943

Avg recall: 0.8128

-- multilingual-e5-large --
  -- arguana --
Minimum recall: 0.9272
Average recall: 0.9311
Maximum recall: 0.9366

  -- nfcorpus --
Minimum recall: 0.6896
Average recall: 0.6937
Maximum recall: 0.6957

Avg recall: 0.8124

-- bge-small-en --
  -- arguana --
Minimum recall: 0.9326
Average recall: 0.9364
Maximum recall: 0.9402

  -- nfcorpus --
Minimum recall: 0.6202
Average recall: 0.6289
Maximum recall: 0.6393

Avg recall: 0.7826

-- stella-base-en-v2 --
  -- arguana --
Minimum recall: 0.9358
Average recall: 0.9399
Maximum recall: 0.9424

  -- nfcorpus --
Minimum recall: 0.7221
Average recall: 0.7252
Maximum recall: 0.7287

Avg recall: 0.8326

-- all-MiniLM-L6-v2 --
  -- arguana --
Minimum recall: 0.9769
Average recall: 0.9785
Maximum recall: 0.9825

  -- 

In [74]:
print(df_bm)
# df_bm.loc[df_bm.em == em, 'lid_d'] = 155
# print(df_bm)
# df_bm_avg = df_bm.groupby(['em'])['random', 'lid_d'].mean()
df_bm_avg = df_bm.groupby("em", as_index=False)[["random", "lid_d", "lid_a"]].mean()
print(df_bm_avg)

                      em        dn    random     lid_d     lid_a
0            e5-small-v2   arguana  0.914206  0.949743  0.933447
1            e5-small-v2  nfcorpus  0.697569  0.744339  0.692133
2  multilingual-e5-large   arguana  0.915218  0.956984  0.931101
3  multilingual-e5-large  nfcorpus  0.704213  0.749696  0.693650
4           bge-small-en   arguana  0.920912  0.965584  0.936362
5           bge-small-en  nfcorpus  0.643827  0.756968  0.628902
6      stella-base-en-v2   arguana  0.916435  0.960726  0.939937
7      stella-base-en-v2  nfcorpus  0.742558  0.807731  0.725178
8       all-MiniLM-L6-v2   arguana  0.955996  0.987565  0.978544
9       all-MiniLM-L6-v2  nfcorpus  0.782957  0.824629  0.761695
                      em    random     lid_d     lid_a
0       all-MiniLM-L6-v2  0.869477  0.906097  0.870119
1           bge-small-en  0.782369  0.861276  0.782632
2            e5-small-v2  0.805888  0.847041  0.812790
3  multilingual-e5-large  0.809715  0.853340  0.812376
4      ste

In [75]:
df_filename = "df_bm_v1"
df_output_path = os.path.join("..\\benchmark\\", df_filename + ".csv")
df_bm.to_csv(df_output_path)

df_filename = "df_bm_avg_v1"
df_output_path = os.path.join("..\\benchmark\\", df_filename + ".csv")
df_bm_avg.to_csv(df_output_path)